In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# 1. 라이브러리 설치 및 환경 설정
!pip install evaluate
!pip install -q transformers datasets peft accelerate
!pip install sacrebleu
!pip install bert_score
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
import numpy as np
import evaluate
from transformers import DataCollatorForSeq2Seq
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00


In [6]:
# GPU 사용 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

사용 장치: cuda


In [7]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

train_size = len(pd.read_csv('./data/final_train.csv'))
valid_size = len(pd.read_csv('./data/final_valid.csv'))
test_size = len(pd.read_csv('./data/final_test.csv'))

train_dataset = load_dataset('csv', data_files='./data/final_train.csv', split='train', streaming=True)
valid_dataset = load_dataset('csv', data_files='./data/final_valid.csv', split='train', streaming=True)
test_dataset = load_dataset('csv', data_files='./data/final_test.csv', split='train', streaming=True)


print(f"훈련 데이터 크기: {train_size}")
print(f"검증 데이터 크기: {valid_size}")
print(f"테스트 데이터 크기: {test_size}")

훈련 데이터 크기: 1567943
검증 데이터 크기: 257002
테스트 데이터 크기: 257004


In [ ]:
# !pip install --upgrade transformers

In [8]:
# 4. 토크나이저 및 모델 로드
model_name = "gogamza/kobart-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# 5. 양방향 학습을 위한 토큰 추가
# '제주'와 '표준' 토큰을 토크나이저에 추가하고 모델 임베딩 크기를 조정합니다.
tokenizer.add_special_tokens({'additional_special_tokens': ['[제주]', '[표준]', '[복원]']})
model.resize_token_embeddings(len(tokenizer))
print(f"모델 임베딩 크기: {len(tokenizer)}")

# 7. 토큰화 함수 정의 및 적용
def tokenize_function(examples):
    model_inputs = tokenizer(examples['input_text'], truncation=True)
    labels = tokenizer(text_target=examples['target_text'], truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_valid_dataset = valid_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# 8. 평가 지표 계산 함수 정의
metric_bleu = evaluate.load("bleu")
metric_chrf = evaluate.load("chrf")
# metric_bertscore = evaluate.load("bertscore")
# metric_comet = evaluate.load("comet")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # perplexity = math.exp(metrics["eval_loss"])

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu_result = metric_bleu.compute(predictions=decoded_preds, references=[[label] for label in decoded_labels])
    chrf_result = metric_chrf.compute(predictions=decoded_preds, references=decoded_labels)
    # bertscore_result = metric_bertscore.compute(predictions=decoded_preds, references=decoded_labels, lang="ko")
    # comet_result = metric_comet.compute(predictions=decoded_preds, references=decoded_labels)

    result = {
        "bleu": bleu_result["bleu"],
        "chrf": chrf_result["score"],
        # "bertscore_f1": np.mean(bertscore_result["f1"]),
        # "perplexity": perplexity
    }

    return result

# 9. 학습 파라미터 및 Trainer 설정
training_args = Seq2SeqTrainingArguments(
    output_dir="./lora_bidirectional_model",
    eval_strategy="epoch",
    learning_rate=5e-6,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=5,
    weight_decay=0.005,
    logging_dir='./logs_full',
    logging_steps=500,
    save_strategy="epoch",
    report_to='none',
    fp16=True,
    gradient_accumulation_steps=16,
    predict_with_generate=True,
    max_grad_norm=1.0, # 그라디언트 클리핑 적용
    label_smoothing_factor=0.1, # 레이블 스무딩 적용
)

steps_per_epoch = train_size / training_args.per_device_train_batch_size
steps_per_epoch = math.ceil(steps_per_epoch / training_args.gradient_accumulation_steps)
max_steps = int(steps_per_epoch * training_args.num_train_epochs)
training_args.max_steps = max_steps

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

class WeightedLossTrainer(Seq2SeqTrainer):
    # 'num_items_in_batch' 인자 제거
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # self.processing_class를 사용
        end_token_id = self.processing_class.eos_token_id

        loss_fct = torch.nn.CrossEntropyLoss(weight=torch.tensor(
            [1.0] * len(self.processing_class)).to(model.device), ignore_index=-100)

        loss_fct.weight[end_token_id] = 2.0

        logits = logits[..., :-1, :].contiguous()
        labels = labels[..., 1:].contiguous()

        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_valid_dataset,
    processing_class=tokenizer, # 여기서 tokenizer를 명시적으로 전달
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 10. 학습 시작
print("--- 양방향 모델 학습 시작 ---")
trainer.train()
print("--- 학습 완료 ---")

# 11. 모델 저장
trainer.save_model("./model/full_finetuned_bidirectional_model_003")
print("--- 학습된 모델 저장 완료 ---")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

...

--- 양방향 모델 학습 시작 ---


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss,Bleu,Chrf
0,3.178900,9.953370,0.020073,12.528596
1,1.565100,11.082689,0.019975,12.842043
2,1.448300,11.516256,0.019643,12.756114


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3917: UserWarning: Moving the following attributes in the config to the generation config: {'forced_eos_token_id': 1}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
 # 12. 태스크별 성능 평가 (수정)
import pandas as pd
from datasets import Dataset

# 테스트 데이터셋 로드
test_df = pd.read_csv('./data/final_test.csv')

# '제주 방언 -> 표준어' 태스크 데이터 필터링
jeju_to_std_df = test_df[test_df['input_text'].str.startswith('[제주]')].copy()
jeju_to_std_dataset = Dataset.from_pandas(jeju_to_std_df)

# '표준어 -> 제주 방언' 태스크 데이터 필터링
std_to_jeju_df = test_df[test_df['input_text'].str.startswith('[표준]')].copy()
std_to_jeju_dataset = Dataset.from_pandas(std_to_jeju_df)

# '복원' 태스크 데이터 필터링
restore_df = test_df[test_df['input_text'].str.startswith('[복원]')].copy()
restore_dataset = Dataset.from_pandas(restore_df)

# 태스크별 데이터셋 토큰화
tokenized_jeju_to_std_test = jeju_to_std_dataset.map(tokenize_function, batched=True)
tokenized_std_to_jeju_test = std_to_jeju_dataset.map(tokenize_function, batched=True)
tokenized_restore_test = restore_dataset.map(tokenize_function, batched=True)

# 13. 태스크별 예측 및 평가 실행 (그리디 서치)
print("\n--- 그리디 서치(Greedy Search) 평가 ---")
print("  - 모델의 가장 순수한 성능 확인")

print("\n  > 제주 방언 → 표준어")
jeju_to_std_preds_greedy = trainer.predict(
    test_dataset=tokenized_jeju_to_std_test,
    predict_with_generate=True,
    **{"num_beams": 1}
)
jeju_to_std_results_greedy = compute_metrics(
    (jeju_to_std_preds_greedy.predictions, jeju_to_std_preds_greedy.label_ids)
)
print(jeju_to_std_results_greedy)


print("\n  > 표준어 → 제주 방언")
std_to_jeju_preds_greedy = trainer.predict(
    test_dataset=tokenized_std_to_jeju_test,
    predict_with_generate=True,
    **{"num_beams": 1}
)
std_to_jeju_results_greedy = compute_metrics(
    (std_to_jeju_preds_greedy.predictions, std_to_jeju_preds_greedy.label_ids)
)
print(std_to_jeju_results_greedy)

print("\n  > 복원 태스크")
restore_preds_greedy = trainer.predict(
    test_dataset=tokenized_restore_test,
    predict_with_generate=True,
    **{"num_beams": 1}
)
restore_results_greedy = compute_metrics(
    (restore_preds_greedy.predictions, restore_preds_greedy.label_ids)
)
print(restore_results_greedy)


# 14. 태스크별 예측 및 평가 실행 (빔 서치)
print("\n--- 빔 서치(Beam Search) 평가 ---")
print("  - 최적화된 문장 탐색")

print("\n  > 제주 방언 → 표준어")
jeju_to_std_preds_beam = trainer.predict(
    test_dataset=tokenized_jeju_to_std_test,
    predict_with_generate=True,
    **{"num_beams": 5, "no_repeat_ngram_size": 3}
)
jeju_to_std_results_beam = compute_metrics(
    (jeju_to_std_preds_beam.predictions, jeju_to_std_preds_beam.label_ids)
)
print(jeju_to_std_results_beam)

print("\n  > 표준어 → 제주 방언")
std_to_jeju_preds_beam = trainer.predict(
    test_dataset=tokenized_std_to_jeju_test,
    predict_with_generate=True,
    **{"num_beams": 5, "no_repeat_ngram_size": 3}
)
std_to_jeju_results_beam = compute_metrics(
    (std_to_jeju_preds_beam.predictions, std_to_jeju_preds_beam.label_ids)
)
print(std_to_jeju_results_beam)

print("\n  > 복원 태스크")
restore_preds_beam = trainer.predict(
    test_dataset=tokenized_restore_test,
    predict_with_generate=True,
    **{"num_beams": 5, "no_repeat_ngram_size": 3}
)
restore_results_beam = compute_metrics(
    (restore_preds_beam.predictions, restore_preds_beam.label_ids)
)
print(restore_results_beam)

- batch() : 입력 리스트에 대한 chain을 호출하는 함수